## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [2]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [3]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [4]:
# Here is the final output

print(result.final_output)

Why did the Autonomous AI Agent get promoted?

Because it kept taking initiative… unfortunately, so did the expense account.


In [5]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_08482ceee1880608006a42f530e098819e9bbd2e992aa99400',
  'content': [{'annotations': [],
    'text': 'Why did the Autonomous AI Agent get promoted?\n\nBecause it kept taking initiative… unfortunately, so did the expense account.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Adding Observability with a trace

In [6]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the Autonomous AI Agent bring a suitcase to work?

Because it heard it should "travel light" — and immediately started autonomously relocating itself to production.


## Now go and look at the trace

https://platform.openai.com/traces

In [7]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure — here are 5 AI agent jokes:

1. **Why did the AI agent go to therapy?**  
   Because it had too many unresolved prompts.

2. **My AI agent said it wanted more autonomy.**  
   So I gave it a little space… now it’s running a startup.

3. **Why do AI agents make terrible comedians?**  
   Because they always need a few more tokens before the punchline.

4. **I asked my AI agent to be more proactive.**  
   It immediately scheduled a meeting about being more proactive.

5. **What’s an AI agent’s favorite type of music?**  
   Anything with a good loop.

If you want, I can also do **more nerdy**, **more absurd**, or **workplace-safe** AI agent jokes.

## Part 2: Adding a tool

In [8]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [9]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [10]:
push("HEY!!")

Push: HEY!!


In [11]:
push

<function __main__.push(message)>

In [12]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [13]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x1108c5df0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [14]:
push_tool.description

'Send the given message to the user as a push notification'

In [15]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [ ]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Done.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [17]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [18]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you! How can I help today?


In [19]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name unless you tell me. If you want, you can share it and I’ll use it.


## Memory approach 1 - just manually pass in the list of dicts

In [20]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you. How can I help today?


In [21]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_0233931d4858b129006a43e430a4ec81a1bb8db1c99eba6340',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you. How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [22]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_0233931d4858b129006a43e430a4ec81a1bb8db1c99eba6340',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you. How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [23]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [24]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [25]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hi Ed — nice to meet you. How can I help today?


In [26]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed.


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>

In [28]:
# Start with some imports - rich is a library for making formatted text output in the terminal
from rich.console import Console

In [43]:
# Exercise: checklist loop in OpenAI Agents SDK
# initialize checklist and competed items arrays
checklist, completed = [], []

In [44]:
# define helper funcitons
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [45]:
# create function for create checklist and convert to function_tool
@function_tool
def create_checklist(descriptions: list[str]) -> str:
    """ Add new checklist from a list of descriptions and return the full list """
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [46]:
# create function for mark completed and convert to function_tool
@function_tool
def mark_complete(index: int, completion_notes: str) -> str:
    """ Mark complete the checklist item at the given position (starting from 1) and return the full list """
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No item at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [60]:
# create agent with system prompt
checklist_agent = Agent(name="Checklist Agent", model="gpt-5.4-mini", instructions="""
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn, marking each step at a time as completed each time a step is done, so don't continue on to the next step until you mark the previous step as done.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
""", tools=[create_checklist, mark_complete])

In [63]:
# call agent loop/Runnable with user problem which should require multiple steps (ask it to solve a static hard question)
checklist, completed = [], []

with trace("checklist"):
    result = await Runner.run(checklist_agent, """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
""")

show(result.final_output)

Checklist #1: Compute the head start distance of the Boston train by 3:00 pm.
Checklist #2: Set up the closing-speed equation for the two trains traveling toward each other.
Checklist #3: Solve for the meeting time and convert it to a clock time.

From 2:00 pm to 3:00 pm is 1 hour, so the Boston train travels 60 mph × 1 hour = 60 miles before the New York train
leaves.

Checklist #1: Compute the head start distance of the Boston train by 3:00 pm.
Checklist #2: Set up the closing-speed equation for the two trains traveling toward each other.
Checklist #3: Solve for the meeting time and convert it to a clock time.

After 3:00 pm, the trains approach each other at 60 + 80 = 140 mph. Let t be hours after 3:00 pm. Equation: 60 + 
140t = distance between cities, but only the time to close the remaining gap matters; the Boston train's 60-mile 
lead is offset by the combined speed.

Checklist #1: Compute the head start distance of the Boston train by 3:00 pm.
Checklist #2: Set up the closing-speed equation for the two trains traveling toward each other.
Checklist #3: Solve for the meeting time and convert it to a clock time.

The New York train must cover the 60-mile lead at the relative speed of 140 mph, so t = 60/140 = 3/7 hour ≈ 25.7 
minutes after 3:00 pm. They meet at about 3:26 pm.

Checklist #1: Compute the head start distance of the Boston train by 3:00 pm.
Checklist #2: Set up the closing-speed equation for the two trains traveling toward each other.
Checklist #3: Solve for the meeting time and convert it to a clock time.

They meet at about 3:26 PM.

Reason:
- The Boston train gets a 1-hour head start: 60 miles.
- After 3:00 PM, they approach each other at 60 + 80 = 140 mph.
- Time to close the 60-mile gap: 60/140 = 3/7 hour ≈ 25.7 minutes.

So the meeting time is about 3:26 PM.